# 01 · Patch expression object — transcripts binned onto the H&E patch grid, 7 donors

**What this notebook does.** Every QV ≥ 20 transcript of each Xenium section is mapped through the
H&E ↔ Xenium affine alignment onto the 56 × 56 µm H&E patch grid (224 px at 0.25 µm/px — the same
grid the morphology niches were defined on) and counted per patch × gene. The result is a
segmentation-free expression object with one row per tissue patch and the morphological niche
label of that patch.

**What is new.** U48 and U51 (the two infant sections) now have Xenium zarr containers, so all
seven donors are built from transcripts. Their run used a later panel version with **50 extra
genes**; those are kept in the object but flagged in `var` (`v2_only`) and are **excluded from every
score downstream** — panel version is perfectly confounded with age group, so a gene measured only
in infants cannot enter a cross-donor comparison.

**Built-in check.** Each patch's transcript count is compared with the sum of the segmented cells
that fall in that patch (cells carry their `patchID` from the same alignment). If the affine were
wrong for a section, the per-patch correlation would collapse — so this is the alignment test for
the two new sections.

| input | what |
|---|---|
| `thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad` (obs only) | H&E patch grid: tile id, `x0/y0`, `leiden_r2.0` |
| `morpho_niches.xlsx` | `leiden_r2.0` → `morpho_niche_l1/l2/l3`, `morpho_niche_keep` |
| `he_alignments/{donor}_alignment_files/matrix.csv` | H&E → Xenium affine (inverted here) |
| `*_xenium.zarr/points/transcripts/points.parquet` | transcripts (x, y in µm, `feature_name`, `qv`) |
| `xenium_aging_labels.h5ad` + `thyAging_xenium_morpho_niches.csv` | segmented cells + their `patchID`, for the check |

**Output** `data/xenium/patchExpr/thyAging_xenium_patchExpr_7donors_<date>.h5ad` — `X` = integer
transcript counts (patches × 5,051 features), `obs` = donor / age / grid position / niche labels /
`keep`, `var` = per-donor detection flags.

Runs in ~20–30 min as an LSF job (`_jobs/run_nb.sh 01`). Nothing here needs a GPU or spatialdata:
the parquet parts are read directly with pyarrow.

In [1]:
import os
import glob
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
import anndata as ad
from anndata.io import sparse_dataset
import pyarrow.parquet as pq
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings('ignore', category=FutureWarning)
today = datetime.now().strftime('%Y-%m-%d')

REPO = '/nfs/team205/vk8/projects/thymus_ageing_atlas'
SPATIAL = f'{REPO}/Spatial_analyses'
GENERAL = f'{REPO}/General_analysis'
LM = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses'

font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
plt.style.use(f'{GENERAL}/scripts/plotting/thyAgeing.mplstyle')
pd.set_option('display.width', 200)

# ---- inputs -------------------------------------------------------------------------------
HE_H5AD = f'{LM}/data/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad'
NICHE_XLSX = f'{LM}/data/morpho_niches/morpho_niches.xlsx'
ALIGN_DIR = f'{SPATIAL}/data/xenium/he_alignments'
CELL_H5AD = f'{SPATIAL}/data/xenium/xenium_proseg/xenium_aging_labels.h5ad'
MORPHO_CSV = f'{SPATIAL}/data/xenium/thyAging_xenium_morpho_niches.csv'
ZARRS = {'A67': f'{SPATIAL}/data/xenium/A67-THY-0-FFPE-1_xenium.zarr',
         'A70': f'{SPATIAL}/data/xenium/A70-THY-0-FFPE-2_xenium.zarr',
         'A72': f'{SPATIAL}/data/xenium/A72-THY-0-FFPE-1_xenium.zarr',
         'Z16': f'{SPATIAL}/data/xenium/Z16-THY-0-FF-1_xenium.zarr',
         'Z17': f'{SPATIAL}/data/xenium/Z17-THY-0-FF-1_xenium.zarr',
         'U48': f'{SPATIAL}/notebooks/data/xenium/U48_xenium.zarr',
         'U51': f'{SPATIAL}/notebooks/data/xenium/U51_xenium.zarr'}

# ---- outputs ------------------------------------------------------------------------------
OUT_DIR = f'{SPATIAL}/data/xenium/patchExpr'
RESULTS = f'{SPATIAL}/results/xenium/final/01_patch_object'
PLOTS = f'{SPATIAL}/plots/xenium/final/01_patch_object'
for d in (OUT_DIR, RESULTS, PLOTS):
    os.makedirs(d, exist_ok=True)
OUT_H5AD = f'{OUT_DIR}/thyAging_xenium_patchExpr_7donors_{today}.h5ad'

# ---- donors and geometry ------------------------------------------------------------------
DONOR_AGE = {'U51': 'epaed', 'U48': 'epaed', 'Z16': 'lpaed', 'Z17': 'lpaed',
             'A67': 'adult', 'A72': 'adult', 'A70': 'adult'}          # A45 failed QC: not built
DONOR_AGE_YEARS = {'U51': 0.04, 'U48': 0.5, 'Z16': 8.0, 'Z17': 11.0, 'A67': 20.0, 'A72': 33.0, 'A70': 37.0}
DONOR_ORDER = ['U51', 'U48', 'Z16', 'Z17', 'A67', 'A72', 'A70']

PIXEL_SIZE = 0.2125        # Xenium µm per pixel (transcript x/y are in µm)
HE_PIXEL_SIZE = 0.221      # H&E scan µm per native pixel
MPP = 0.25                 # µm per pixel of the patch grid
PATCH_PX = 224             # patch edge in grid pixels -> 56 µm
SCALE = HE_PIXEL_SIZE / MPP
QV_MIN = 20
MORPHO_L2_ORDER = ['cortex_septal', 'cortex', 'medulla', 'medulla_hc', 'adipose', 'fibrosis_adipose', 'fibrosis', 'vessel']
MORPHO_L3_ORDER = ['cortex_septal', 'cortex_outer', 'cortex_dense', 'cortex', 'cortex_residual',
                   'medulla', 'medulla_hc', 'medulla_residual', 'adipose', 'adipose_edge',
                   'fibrosis', 'fibrosis_adipose', 'endothelium_vessel', 'endothelium_edge']
print(f'{today} | output -> {OUT_H5AD}')

2026-09-04 | output -> /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/data/xenium/patchExpr/thyAging_xenium_patchExpr_7donors_2026-09-04.h5ad


## 1 · The patch grid

Only the `obs` of the H&E object is read (the file is 5.8 GB, the embeddings are not needed).
Patches are the `.ome` sections of the seven donors; `patch_col/row` = `x0/y0 // 224`. Niche labels
come from the cluster → niche table.

In [2]:
def read_obs_h5(path, cols):
    """A few obs columns straight from HDF5 (categoricals decoded). Avoids anndata version issues."""
    out = {}
    with h5py.File(path, 'r') as f:
        g = f['obs']
        idx = pd.Index(np.asarray(g[g.attrs['_index']][:], dtype='S').astype(str))
        for c in cols:
            e = g[c]
            if isinstance(e, h5py.Group):
                cats = np.asarray(e['categories'][:], dtype='S').astype(str)
                out[c] = pd.Categorical.from_codes(e['codes'][:], categories=list(cats))
            else:
                v = e[:]
                out[c] = np.asarray(v, dtype='S').astype(str) if v.dtype.kind in 'SO' else v
    return pd.DataFrame(out, index=idx)


he = read_obs_h5(HE_H5AD, ['SampleID', 'DonorID', 'has_xenium', 'x0', 'y0', 'x1', 'y1', 'leiden_r2.0'])
he['SampleID'] = he['SampleID'].astype(str)
he = he[he['SampleID'].isin([f'{d}.ome' for d in DONOR_AGE])].copy()
he['donor'] = he['SampleID'].str.replace('.ome', '', regex=False)
he['patch_col'] = (he['x0'] // PATCH_PX).astype(int)
he['patch_row'] = (he['y0'] // PATCH_PX).astype(int)
assert not he.duplicated(['SampleID', 'patch_col', 'patch_row']).any(), 'duplicate grid positions'

niche = pd.read_excel(NICHE_XLSX, index_col=0)[['morpho_niche_l3', 'morpho_niche_l2', 'morpho_niche_l1', 'morpho_niche_keep']]
niche.index = niche.index.astype(str)
lab = niche.reindex(he['leiden_r2.0'].astype(str).to_numpy())
lab.index = he.index
he = pd.concat([he.drop(columns=[c for c in niche.columns if c in he.columns]), lab], axis=1)
assert he['morpho_niche_keep'].notna().all(), 'a leiden_r2.0 cluster has no niche label'
he['morpho_niche_keep'] = he['morpho_niche_keep'].astype(bool)
he['age_group'] = he['donor'].map(DONOR_AGE)
he['age_years'] = he['donor'].map(DONOR_AGE_YEARS)
he = he.sort_values(['donor', 'patch_row', 'patch_col'])
he = he.loc[sum([list(he.index[he['donor'] == d]) for d in DONOR_ORDER], [])]   # donor order fixed

print(f'{len(he):,} patches on the seven Xenium sections')
print(he.groupby('donor').agg(n_patches=('x0', 'size'), n_keep=('morpho_niche_keep', 'sum'),
                              grid_cols=('patch_col', 'max'), grid_rows=('patch_row', 'max')).reindex(DONOR_ORDER))
print()
print(pd.crosstab(he['donor'], he['morpho_niche_l2']).reindex(DONOR_ORDER).reindex(columns=MORPHO_L2_ORDER + [c for c in he['morpho_niche_l2'].unique() if c not in MORPHO_L2_ORDER]))

95,830 patches on the seven Xenium sections
       n_patches  n_keep  grid_cols  grid_rows
donor                                         
U51        12924   12045        119        187
U48        15760   14655        169        135
Z16         3627    3138         80         85
Z17         7572    7434         81        141
A67        12324    9981        111        187
A72        23494   21565        187        212
A70        20129   17840        141        227

morpho_niche_l2  cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel  artifact
donor                                                                                                             
U51                        802    8149     1865         101       84               751       235      58       879
U48                        790    8789     4198         188      116                94       248     232      1105
Z16                        363    1422      884          84       51    

## 2 · Transcripts → patches

Two passes per donor over the parquet parts. Pass 1 collects the feature names (this is how the
panel of each run is learned — no zarr reader needed); pass 2 bins the transcripts. Feature names
that are controls / unassigned codewords are dropped and listed. The mapping is

    µm  →  Xenium px (÷ 0.2125)  →  H&E native px (inverse affine)  →  grid px (× 0.221/0.25)  →  patch (÷ 224)

exactly as in the original construction (`05a_patchExpression`).

In [3]:
def parquet_parts(zarr_path):
    d = f'{zarr_path}/points/transcripts/points.parquet'
    return sorted(glob.glob(f'{d}/*.parquet')) if os.path.isdir(d) else [d]


def apply_affine(M, xy):
    h = np.hstack([xy, np.ones((len(xy), 1))])
    return (h @ M.T)[:, :2]


CONTROL_RE = r'^(NegControl|Unassigned|Deprecated|BLANK|Genomic|Intergenic|antisense)'   # non-gene features

# pass 1: feature names per donor
feat = {}
t0 = time.time()
for d in DONOR_ORDER:
    names = set()
    for part in parquet_parts(ZARRS[d]):
        col = pq.read_table(part, columns=['feature_name']).column('feature_name')
        names |= set(pd.unique(col.to_pandas().astype(str)))
    feat[d] = names
    print(f'{d}: {len(parquet_parts(ZARRS[d]))} parts, {len(names):,} distinct feature names  ({time.time() - t0:.0f}s)')

all_names = pd.Index(sorted(set().union(*feat.values())))
all_names = all_names[~all_names.isin(['nan', 'None', ''])]        # a missing feature_name reads as 'nan'
is_ctrl = all_names.str.match(CONTROL_RE)
GENES = all_names[~is_ctrl]
print(f'\n{len(all_names):,} feature names in total; {int(is_ctrl.sum())} controls dropped, {len(GENES):,} genes kept')
print('dropped:', ', '.join(sorted(set(n.split('_')[0] for n in all_names[is_ctrl]))))
n_donors = pd.Series({g: sum(g in feat[d] for d in DONOR_ORDER) for g in GENES})
print('genes by number of donors whose run contains them:', n_donors.value_counts().sort_index().to_dict())
extra = sorted(n_donors.index[n_donors < len(DONOR_ORDER)])
print(f'not in every run ({len(extra)}):', ', '.join(extra))

U51: 19 parts, 8,915 distinct feature names  (11s)


U48: 56 parts, 8,700 distinct feature names  (45s)


Z16: 26 parts, 6,485 distinct feature names  (52s)


Z17: 60 parts, 7,047 distinct feature names  (69s)


A67: 19 parts, 6,171 distinct feature names  (75s)


A72: 142 parts, 6,599 distinct feature names  (114s)


A70: 19 parts, 8,320 distinct feature names  (120s)

10,009 feature names in total; 4958 controls dropped, 5,051 genes kept
dropped: DeprecatedCodeword, Intergenic, NegControlCodeword, NegControlProbe, UnassignedCodeword
genes by number of donors whose run contains them: {1: 1, 2: 50, 4: 4, 5: 6, 6: 68, 7: 4922}
not in every run (129): ACKR1, ANXA1, APOC2, AQP1, AQP5, ATP6AP1L, BACH1, BHMT, CA10, CCL2, CCL3, CCL5, CDX2, CFAP46, CHRM1, CHRNA1, CHRNA4, COLEC11, CRH, CSF2, CSF3, CST1, CXCR2, CYP11B2, CYP2A6, DCT, DCX, DES, ELOVL4, EN2, EPHA8, EPO, F2, FEZF1, FGF21, FUT2, FXYD3, GAD2, GDF3, GJB1, GNG4, GNLY, GPR158, GRHL1, GRIN2B, HBA2, HCN4, HIVEP3, HLA-DRA, HMX2, HNF1A, HPD, HRH3, HSD11B1, IGFBP1, IGFBP6, IGLL1, IVL, KCNQ2, KLF2, KLK4, KRT1, KRT14, KRT15, KRT20, KRT5, KRT6A, KRT8, LBP, LCN2, LMNA, LRRTM1, LUM, LYVE1, MAGEA1, MFRP, MUSK, MYH11, MYOD1, MYOG, NCR2, NLRP5, NLRP7, OLIG1, ONECUT3, OPN1LW, PAEP, PCDH11Y, PCDHA2, PCK1, PCP4, PDIA2, PITX1, PLG, POU3F4, PROKR2, PSMB11, PTCRA, RPRM

In [4]:
def bin_donor(d, genes):
    """Dense patch x gene counts for one donor (its own patches, in `he` order), plus a stats row."""
    sub = he[he['donor'] == d]
    grid = np.full((sub['patch_row'].max() + 1, sub['patch_col'].max() + 1), -1, dtype=np.int64)
    grid[sub['patch_row'].to_numpy(), sub['patch_col'].to_numpy()] = np.arange(len(sub))
    M = pd.read_csv(f'{ALIGN_DIR}/{d}_alignment_files/matrix.csv', header=None).values
    Minv = np.linalg.inv(M)
    n_gene = len(genes)
    acc = np.zeros((len(sub), n_gene), dtype=np.int32)
    n_seen = n_gene_tx = n_in = 0
    cols = None
    for part in parquet_parts(ZARRS[d]):
        if cols is None:
            schema = pq.read_schema(part).names
            assert all(c in schema for c in ['x', 'y', 'feature_name', 'qv']), schema
            cols = ['x', 'y', 'feature_name', 'qv']
        df = pq.read_table(part, columns=cols).to_pandas()
        df = df[df['qv'].to_numpy() >= QV_MIN]
        if not len(df):
            continue
        cat = df['feature_name'].astype(str).astype('category')
        cmap = genes.get_indexer(cat.cat.categories)
        g = cmap[cat.cat.codes.to_numpy()]
        xy = apply_affine(Minv, df[['x', 'y']].to_numpy(np.float64) / PIXEL_SIZE) * SCALE
        col = np.floor(xy[:, 0] / PATCH_PX).astype(np.int64)
        row = np.floor(xy[:, 1] / PATCH_PX).astype(np.int64)
        inb = (g >= 0) & (row >= 0) & (col >= 0) & (row < grid.shape[0]) & (col < grid.shape[1])
        p = np.full(len(df), -1, dtype=np.int64)
        p[inb] = grid[row[inb], col[inb]]
        ok = p >= 0
        lin = p[ok] * n_gene + g[ok]
        u, c = np.unique(lin, return_counts=True)
        acc.flat[u] += c.astype(np.int32)
        n_seen += len(df)
        n_gene_tx += int((g >= 0).sum())
        n_in += int(ok.sum())
    stats = {'donor': d, 'n_transcripts_qv20': n_seen, 'n_gene_transcripts': n_gene_tx,
             'n_in_tissue_patches': n_in, 'frac_in_tissue_patches': n_in / max(n_gene_tx, 1)}
    return acc, stats


blocks, stat_rows = [], []
t0 = time.time()
for d in DONOR_ORDER:
    acc, st = bin_donor(d, GENES)
    blocks.append(sp.csr_matrix(acc))
    stat_rows.append(st)
    del acc
    print(f'{d}: {st["n_gene_transcripts"]:,} gene transcripts (QV>={QV_MIN}), '
          f'{st["frac_in_tissue_patches"]:.1%} inside tissue-mask patches  ({time.time() - t0:.0f}s)', flush=True)

X = sp.vstack(blocks).tocsr()
del blocks
tx_stats = pd.DataFrame(stat_rows).set_index('donor')
assert X.shape == (len(he), len(GENES))
print(f'\npatch matrix {X.shape[0]:,} x {X.shape[1]:,}, {X.nnz:,} non-zeros, {int(X.sum()):,} transcripts')

U51: 92,581,613 gene transcripts (QV>=20), 100.0% inside tissue-mask patches  (33s)


U48: 333,741,455 gene transcripts (QV>=20), 100.0% inside tissue-mask patches  (152s)


Z16: 68,684,172 gene transcripts (QV>=20), 99.6% inside tissue-mask patches  (179s)


Z17: 158,148,487 gene transcripts (QV>=20), 100.0% inside tissue-mask patches  (238s)


A67: 49,208,460 gene transcripts (QV>=20), 99.9% inside tissue-mask patches  (258s)


A72: 366,609,231 gene transcripts (QV>=20), 99.8% inside tissue-mask patches  (366s)


A70: 47,109,143 gene transcripts (QV>=20), 98.5% inside tissue-mask patches  (383s)



patch matrix 95,830 x 5,051, 164,095,724 non-zeros, 1,114,000,163 transcripts


## 3 · Assemble the object

In [5]:
obs = he[['SampleID', 'donor', 'age_group', 'age_years', 'patch_col', 'patch_row', 'x0', 'y0', 'x1', 'y1',
          'leiden_r2.0', 'morpho_niche_l1', 'morpho_niche_l2', 'morpho_niche_l3', 'morpho_niche_keep']].copy()
obs['leiden_r2.0'] = obs['leiden_r2.0'].astype(str)
for c in ['morpho_niche_l1', 'morpho_niche_l2', 'morpho_niche_l3']:
    obs[c] = obs[c].astype(str)
obs['n_transcripts'] = np.asarray(X.sum(axis=1)).ravel().astype(np.int64)
obs['keep'] = obs['morpho_niche_keep'] & (obs['n_transcripts'] > 0)
obs.index.name = 'patchID'

var = pd.DataFrame(index=pd.Index(GENES, name='gene'))
for d in DONOR_ORDER:
    var[f'in_run_{d}'] = [g in feat[d] for g in GENES]
var['n_runs'] = var[[f'in_run_{d}' for d in DONOR_ORDER]].sum(axis=1).astype(int)
var['in_all_runs'] = var['n_runs'] == len(DONOR_ORDER)
V1_RUNS = ['A67', 'A70', 'A72', 'Z16', 'Z17']                       # sections run on the 5,001-gene v1 panel
var['in_v1_panel'] = var[[f'in_run_{d}' for d in V1_RUNS]].any(axis=1)   # detected in at least one v1 run = on the v1 panel
var['v2_only'] = ~var['in_v1_panel'] & var[['in_run_U48', 'in_run_U51']].any(axis=1)
var['total_counts'] = np.asarray(X.sum(axis=0)).ravel().astype(np.int64)

patch = ad.AnnData(X=X, obs=obs, var=var)
patch.uns['build'] = {'date': today, 'qv_min': QV_MIN, 'pixel_size_um': PIXEL_SIZE, 'he_pixel_size_um': HE_PIXEL_SIZE,
                      'grid_mpp': MPP, 'patch_px': PATCH_PX, 'patch_um': PATCH_PX * MPP, 'donors': DONOR_ORDER}
print(patch)
print(f'\nv1 panel {int(var["in_v1_panel"].sum()):,} genes; v2-only (U48/U51 runs only, excluded from all scoring) {int(var["v2_only"].sum())}; detected in every run {int(var["in_all_runs"].sum()):,}')
print(obs.groupby('donor').agg(n_patches=('keep', 'size'), n_keep=('keep', 'sum'),
                               transcripts=('n_transcripts', 'sum'), median_per_patch=('n_transcripts', 'median')).reindex(DONOR_ORDER))

AnnData object with n_obs × n_vars = 95830 × 5051
    obs: 'SampleID', 'donor', 'age_group', 'age_years', 'patch_col', 'patch_row', 'x0', 'y0', 'x1', 'y1', 'leiden_r2.0', 'morpho_niche_l1', 'morpho_niche_l2', 'morpho_niche_l3', 'morpho_niche_keep', 'n_transcripts', 'keep'
    var: 'in_run_U51', 'in_run_U48', 'in_run_Z16', 'in_run_Z17', 'in_run_A67', 'in_run_A72', 'in_run_A70', 'n_runs', 'in_all_runs', 'in_v1_panel', 'v2_only', 'total_counts'
    uns: 'build'

v1 panel 5,001 genes; v2-only (U48/U51 runs only, excluded from all scoring) 50; detected in every run 4,922
       n_patches  n_keep  transcripts  median_per_patch
donor                                                  
U51        12924    7124     92561484            5211.5
U48        15760   14653    333639347           23175.5
Z16         3627    3138     68407768           18966.0
Z17         7572    7433    158083920           22405.0
A67        12324    9980     49157888             810.5
A72        23494   20329    3657420

## 4 · Check: transcript patches vs cell-summed patches

Every segmented cell carries the `patchID` of the tile it falls in. Summing cell counts per patch
gives an independent construction of the same object (minus the transcripts no cell claimed).
Per donor we report the share of transcripts unassigned to cells, the per-gene profile
correlation, and — the alignment test — the per-patch Spearman between the two totals across kept
patches. A misaligned section would show a low per-patch correlation and many cell-bearing patches
without transcripts.

In [6]:
with h5py.File(CELL_H5AD, 'r') as f:
    g = f['obs']
    cell_names = pd.Index(np.asarray(g[g.attrs['_index']][:], dtype='S').astype(str))
    cats = np.asarray(g['donor']['categories'][:], dtype='S').astype(str)
    cell_donor = pd.Categorical.from_codes(g['donor']['codes'][:], categories=list(cats))
    gv = f['var']
    cell_genes = pd.Index(np.asarray(gv[gv.attrs['_index']][:], dtype='S').astype(str))

idx_col = pd.read_csv(MORPHO_CSV, nrows=0).columns[0]
morpho = pd.read_csv(MORPHO_CSV, index_col=0, usecols=[idx_col, 'patchID'])
patch_of_cell = morpho['patchID'].reindex(cell_names)
codes = patch.obs_names.get_indexer(patch_of_cell.fillna('__none__').astype(str))
print(f'{len(cell_names):,} cells; {int((codes >= 0).sum()):,} fall in a patch of this object')

# stream the count layer once; contiguous chunks, sparse accumulation per patch
t0 = time.time()
cell_sum = None
n_cells_patch = np.zeros(patch.n_obs, dtype=np.int64)
with h5py.File(CELL_H5AD, 'r') as f:
    dset = sparse_dataset(f['layers/counts'])
    n_rows = dset.shape[0]
    for start in range(0, n_rows, 200_000):
        stop = min(start + 200_000, n_rows)
        c = codes[start:stop]
        sel = np.flatnonzero(c >= 0)
        if not sel.size:
            continue
        blk = dset[start:stop]
        S = sp.csr_matrix((np.ones(sel.size, dtype=np.float32), (c[sel], sel)), shape=(patch.n_obs, stop - start))
        part = (S @ blk).tocsr()
        cell_sum = part if cell_sum is None else cell_sum + part
        n_cells_patch += np.bincount(c[sel], minlength=patch.n_obs)
        del blk, S, part
print(f'cell sums streamed in {time.time() - t0:.0f}s')
patch.obs['n_cells'] = n_cells_patch

shared = patch.var_names.intersection(cell_genes)
print(f'genes: patch object {patch.n_vars:,}, cell object {len(cell_genes):,}, shared {len(shared):,}')
print('patch-only:', ', '.join(patch.var_names.difference(cell_genes)))
Xt = patch[:, shared].X
Xc = cell_sum[:, cell_genes.get_indexer(shared)]

3,531,679 cells; 3,436,646 fall in a patch of this object


cell sums streamed in 29s
genes: patch object 5,051, cell object 5,043, shared 5,043
patch-only: DEFA1, DEFB103B, GUCA1A, IFNA17, KCNK18, SSX2, SSX4, TSPY1


In [7]:
rows = []
keep = patch.obs['keep'].to_numpy()
for d in DONOR_ORDER:
    m = (patch.obs['donor'] == d).to_numpy() & keep
    tt = np.asarray(Xt[m].sum(axis=1)).ravel()
    cc = np.asarray(Xc[m].sum(axis=1)).ravel()
    gt = np.asarray(Xt[m].sum(axis=0)).ravel()
    gc = np.asarray(Xc[m].sum(axis=0)).ravel()
    rows.append({'donor': d, 'n_kept_patches': int(m.sum()),
                 'tx_counts': float(tt.sum()), 'cell_counts': float(cc.sum()),
                 'frac_unassigned_to_cells': 1 - cc.sum() / tt.sum(),
                 'gene_profile_r': float(np.corrcoef(np.log1p(gt), np.log1p(gc))[0, 1]),
                 'per_patch_spearman': float(spearmanr(tt, cc).statistic),
                 'patches_with_cells_no_tx': int(((cc > 0) & (tt == 0)).sum()),
                 'patches_with_tx_no_cells': int(((tt > 0) & (cc == 0)).sum())})
calib = pd.DataFrame(rows).set_index('donor')
calib = calib.join(tx_stats)
calib.to_csv(f'{RESULTS}/patch_calibration_{today}.csv')
print(calib.round(4).to_string())
assert (calib['per_patch_spearman'] > 0.9).all(), 'a section is misaligned: per-patch correlation < 0.9'
assert calib['frac_unassigned_to_cells'].between(0, 0.3).all(), 'cell sums exceed transcripts or unassigned share > 30%'
print('\nall seven sections: per-patch correlation > 0.9 and 0-30% of transcripts unassigned to cells')

       n_kept_patches    tx_counts  cell_counts  frac_unassigned_to_cells  gene_profile_r  per_patch_spearman  patches_with_cells_no_tx  patches_with_tx_no_cells  n_transcripts_qv20  n_gene_transcripts  n_in_tissue_patches  frac_in_tissue_patches
donor                                                                                                                                                                                                                                                 
U51              7124   90788783.0   81524146.0                    0.1020          0.9983              0.9937                         0                        81            92657274            92581613             92561484                  0.9998
U48             14653  319157378.0  317862523.0                    0.0041          0.9994              0.9923                         0                        64           333972736           333741455            333639347                  0.9997
Z16         

In [8]:
fig, axes = plt.subplots(2, 4, figsize=(183 / 25.4, 92 / 25.4), layout='constrained')
for ax, d in zip(axes.ravel(), DONOR_ORDER):
    m = (patch.obs['donor'] == d).to_numpy() & keep
    tt = np.log10(1 + np.asarray(Xt[m].sum(axis=1)).ravel())
    cc = np.log10(1 + np.asarray(Xc[m].sum(axis=1)).ravel())
    ax.scatter(tt, cc, s=1, lw=0, alpha=0.3, color='0.2', rasterized=True)
    lim = [0, max(tt.max(), cc.max()) * 1.02]
    ax.plot(lim, lim, color='#DC267F', lw=0.5)
    ax.set_title(f'{d} ({DONOR_AGE[d]}, {DONOR_AGE_YEARS[d]:g} y)  ρ = {calib.loc[d, "per_patch_spearman"]:.3f}')
    ax.set_xlabel('log10 transcripts per patch')
    ax.set_ylabel('log10 cell-summed counts')
axes.ravel()[-1].axis('off')
fig.suptitle('Transcript-binned vs cell-summed patch totals (kept patches)')
fig.savefig(f'{PLOTS}/patch_calibration_scatter.pdf', bbox_inches=None)
plt.show()

## 5 · Write

In [9]:
patch.obs['morpho_niche_l2_ordered'] = pd.Categorical(patch.obs['morpho_niche_l2'], categories=MORPHO_L2_ORDER)
patch.obs['morpho_niche_l3_ordered'] = pd.Categorical(patch.obs['morpho_niche_l3'], categories=MORPHO_L3_ORDER)
patch.write_h5ad(OUT_H5AD, compression='gzip')
print(f'written {OUT_H5AD}  ({os.path.getsize(OUT_H5AD) / 1e6:.0f} MB)')

summary = patch.obs.groupby(['age_group', 'donor'], observed=True).agg(
    n_patches=('keep', 'size'), n_keep=('keep', 'sum'), n_cells=('n_cells', 'sum'),
    transcripts=('n_transcripts', 'sum')).reindex(
    [(DONOR_AGE[d], d) for d in DONOR_ORDER])
summary.to_csv(f'{RESULTS}/patch_object_summary_{today}.csv')
summary

written /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/data/xenium/patchExpr/thyAging_xenium_patchExpr_7donors_2026-09-04.h5ad  (290 MB)


n_patches  n_keep  n_cells  transcripts
age_group donor                                         
epaed     U51        12924    7124   501774     92561484
          U48        15760   14653  1032447    333639347
lpaed     Z16         3627    3138   198930     68407768
          Z17         7572    7433   398351    158083920
adult     A67        12324    9980   202985     49157888
          A72        23494   20329   942213    365742002
          A70        20129   15535   159946     46407754